# Tome C4, Chapter 4: Kolmogorov Complexity (Python companion)

Exercises first, solutions at the end. Kolmogorov complexity cannot be computed (Theorem 4.6), but real compressors give upper bounds, and counting gives everything else. We compress periodic strings, digits of $\pi$ and coin flips, look at runs and balance in random strings, count comparisons in sorting, and watch primes refuse to be rare.

*Before each solution, predict what the output will look like.*

In [ ]:
import bz2, lzma, math, random, zlib
from math import comb, log2, lgamma
import numpy as np
import matplotlib.pyplot as plt
from mpmath import mp
from sympy import factorint, prime, primepi

def h(t):
    return 0.0 if t in (0, 1) else -t*log2(t) - (1-t)*log2(1-t)

def pack(bits):                      # a string of '0'/'1' -> bytes, 8 bits per byte
    bits = bits + '0' * (-len(bits) % 8)
    return int(bits, 2).to_bytes(len(bits) // 8, 'big') if bits else b''

def proxy(bits):                     # best of three compressors, in bits: an upper bound on C(x) up to a constant
    data = pack(bits)
    return 8 * min(len(zlib.compress(data, 9)), len(bz2.compress(data, 9)), len(lzma.compress(data, preset=9)))

## Exercises

**P4.1 (★) The box.** Recover the three $32$-bit strings of the opening box: the periodic one, the first $32$ binary digits of $\pi$, and $32$ fair coin flips made with `random.Random(1948)`. Then compare compressed sizes of long versions ($n=2^{17}$ bits) of each.

**P4.2 (★) Counting.** Check the numbers of Exercise 4.2, and plot the bound of Theorem 4.4: the fraction of strings of length $n$ with $C(x)<n-c$ is below $2^{-c}$.

**P4.3 (★★) Balance and runs.** For coin flips of length $n$, compare the deviation $|k-\frac n2|$ and the longest run of zeros with the bounds of Proposition 4.5 and Exercise 4.4.

**P4.4 (★★) Sorting.** Count the comparisons made by merge sort and by Python's sort on random inputs and compare with $\log_2n!$ (Exercise 4.5).

**P4.5 (★★) Bernoulli strings.** For Bernoulli$(p)$ strings, compare the compressed size per bit with $h(p)$, and with the explicit description of Exercise 4.9 ($n$, the number of ones, and an index).

**P4.6 (★) Kraft for integers.** Show numerically how slowly $\sum_{n\le N}\frac1{n\log_2n}$ grows (Exercise 4.6), and compare the lengths $\log_2n+2\log_2\log_2n$ of the $\delta$ code with $\log_2^*n$.

**P4.7 (★★) Primes.** Check the inequality $p_k\le Ak(\log_2k)^2$ of Theorem 4.18: compute $p_k/(k(\log_2k)^2)$ and compare with Chebyshev. Then take random $n$-bit numbers and look at their largest prime factor, which the proof says cannot be small for incompressible numbers.

**P4.8 (★) Berry.** Count the letters of the phrase of Exercise 4.10.

## Solutions

### P4.1

In [ ]:
periodic = '01' * 16
mp.prec = 200
pi_bits = bin(int(mp.pi * 2**30))[2:]          # 2 integer bits + 30 fractional bits
rnd = random.Random(1948)
coin = ''.join(rnd.choice('01') for _ in range(32))
for name, s in (("periodic", periodic), ("pi", pi_bits), ("coin", coin)):
    print("%-8s %s  ones=%d" % (name, s, s.count('1')))

n = 2**17
mp.prec = n + 64
long_pi = bin(int(mp.pi * 2**(n - 2)))[2:]
rnd = random.Random(1948)
long_coin = ''.join(rnd.choice('01') for _ in range(n))
for name, s in (("periodic", '01' * (n // 2)), ("pi", long_pi), ("coin", long_coin)):
    print("%-8s n=%d bits, ones %.4f, best compressor %d bits" % (name, len(s), s.count('1') / len(s), proxy(s)))

Compressors shrink the periodic string to almost nothing but cannot touch $\pi$: its bits are as balanced as the coin's, and no general-purpose compressor finds the short program (a few lines computing $\pi$) that exists. That is Theorem 4.6 in practice: compressors give upper bounds on $C$, never the value.

### P4.2

In [ ]:
print("fraction with C(x) < 990 among strings of length 1000: < 2^-10 = %.6f" % 2**-10)
print("so more than %.2f%% have C(x) >= 990" % (100 * (1 - 2**-10)))
cs = np.arange(0, 21)
plt.figure(figsize=(5, 3))
plt.semilogy(cs, 2.0 ** -cs, 'o-')
plt.xlabel("c"); plt.ylabel("fraction with C(x) < n - c (bound)"); plt.show()

### P4.3

In [ ]:
rng = np.random.default_rng(4)
def longest_zero_run(b):
    best = cur = 0
    for x in b:
        cur = cur + 1 if x == 0 else 0
        best = max(best, cur)
    return best

print("    n   mean|k-n/2|  sqrt(n/(2 pi))  mean longest 0-run  log2 n")
for n in (2**10, 2**14, 2**18):
    devs, runs = [], []
    for _ in range(20):
        b = rng.integers(0, 2, n)
        devs.append(abs(b.sum() - n / 2)); runs.append(longest_zero_run(b))
    print("%7d %10.1f %12.1f %14.1f %12.1f" % (n, np.mean(devs), math.sqrt(n / (2 * math.pi)), np.mean(runs), log2(n)))
n, c = 2**18, 10
c0 = 10                                    # an illustrative value of the constant c_0
print("Proposition bound for n=2^18, c=10, c_0=10: |k - n/2| <= %.0f" % math.sqrt(n * math.log(2) / 2 * (c + 4 * log2(n) + c0)))

The typical deviation is $\sqrt{n/(2\pi)}$ and the longest run is about $\log_2n-1$, both well inside what incompressibility allows. Incompressible strings are exactly as balanced, and as streaky, as coin flips.

### P4.4

In [ ]:
def merge_sort(a, counter):
    if len(a) <= 1:
        return a
    m = len(a) // 2
    L, Rr = merge_sort(a[:m], counter), merge_sort(a[m:], counter)
    out, i, j = [], 0, 0
    while i < len(L) and j < len(Rr):
        counter[0] += 1
        if L[i] < Rr[j]: out.append(L[i]); i += 1
        else: out.append(Rr[j]); j += 1
    return out + L[i:] + Rr[j:]

class Key:
    count = 0
    def __init__(self, v): self.v = v
    def __lt__(self, o): Key.count += 1; return self.v < o.v

for n in (100, 1000, 10000):
    lf = lgamma(n + 1) / math.log(2)
    a = list(range(n)); random.Random(n).shuffle(a)
    cnt = [0]; merge_sort(a, cnt)
    Key.count = 0; sorted(Key(v) for v in a)
    print("n=%5d  log2 n! = %8.0f   merge sort %7d   Python sort %7d   bound log2 n! - 2 log2 n = %8.0f"
          % (n, lf, cnt[0], Key.count, lf - 2 * log2(n)))

### P4.5

In [ ]:
n = 2**16
rng = np.random.default_rng(5)
print("   p    h(p)   compressor/n   explicit/n")
for p in (0.05, 0.1, 0.25, 0.5):
    b = (rng.random(n) < p).astype(int)
    s = ''.join(map(str, b)); k = int(b.sum())
    explicit = (log2(comb(n, k)) + 4 * log2(n)) / n
    print("%.2f  %.4f  %10.4f  %11.4f" % (p, h(p), proxy(s) / n, explicit))

The explicit description of Exercise 4.9 is within $O(\frac{\log n}n)$ of $h(p)$; generic compressors are within a few percent, and never below $h(p)$ in a meaningful way, as Proposition 4.11 says.

### P4.6

In [ ]:
s, marks = 0.0, {}
N = 10**7
ns = np.arange(2, N + 1, dtype=float)
partial = np.cumsum(1 / (ns * np.log2(ns)))
for M in (10, 10**3, 10**5, 10**7):
    print("sum_{n<=%d} 1/(n log2 n) = %.3f   (ln ln N * ln 2 + const: %.3f)" % (M, partial[M - 2], math.log(2) * math.log(math.log(M))))

def logstar(n):
    t, x = 0.0, float(n)
    while True:
        x = log2(x)
        if x <= 0: return t
        t += x

for n in (10, 10**3, 10**6, 10**12):
    print("n=%g: log2 n + 2 log2 log2 n = %.1f, log2* n = %.1f" % (n, log2(n) + 2 * log2(log2(n)), logstar(n)))

The sum grows like $\ln2\cdot\ln\ln N$: it diverges, but it needs $N=10^7$ to pass $4$. So the Kraft argument of Exercise 4.6 only bites for astronomically many $n$, and $\log_2^*n$ is barely smaller than the $\delta$ code for numbers we can write.

### P4.7

In [ ]:
print("     k        p_k     p_k/(k log2^2 k)   p_k/(k ln k)")
for k in (10, 100, 1000, 10**4, 10**5):
    p = prime(k)
    print("%7d %10d %14.4f %16.4f" % (k, p, p / (k * log2(k)**2), p / (k * math.log(k))))
rng = random.Random(7)
print("largest prime factor of random n-bit numbers (median over 200 draws):")
for nbits in (20, 30, 40):
    lp = sorted(max(factorint(rng.getrandbits(nbits) | (1 << (nbits - 1)))) for _ in range(200))
    med = lp[100]
    print("  n=%d: median largest prime factor has %.1f bits; its index k has %.1f bits" % (nbits, log2(med), log2(int(primepi(med)))))

The ratio $p_k/(k(\log_2k)^2)$ goes to $0$, as it must since $p_k\sim k\ln k$; the theorem's bound is weak but true for every $k$ here. The largest prime factor of a random $n$-bit number typically has about $0.6n$ bits (Dickman's constant, $0.624$), and naming its index costs only slightly fewer bits than writing it: the saving that the proof exploits is $\log_2p_k-\log_2k\approx\log_2\ln p_k$.

### P4.8

In [ ]:
phrase = "the smallest positive integer not definable in under sixty letters"
print(sum(ch.isalpha() for ch in phrase), "letters")